# Device Lifecycle Intelligence System
## Customer Churn Prediction and Retention Economics

## Executive Summary

**Client:** Company A (US Telecom Operator)  
**Objective:** Predict churn, rank retention targets, and quantify ROI for proactive device-upgrade campaigns.

| Item | Result |
|------|--------|
| **Champion model** | ValidationWeightedEnsemble |
| **Holdout PR-AUC** | 0.685036 |
| **Holdout ROC-AUC** | 0.695620 |
| **F1** | 0.689465 |
| **Ensemble weights** | OptunaLightGBM 80% · LightGBM 15% · CatBoost 5% |

This notebook loads `telecom/Client.csv` and `telecom/Record.csv`, performs data quality review and exploratory analysis, engineers lifecycle and segmentation features, benchmarks five model families, optimizes LightGBM with Optuna, blends a validation-weighted ensemble, and translates scores into retention ROI scenarios.

## Business Problem

Telecom operators face rising acquisition costs and saturated markets. For Company A, **device obsolescence** (`eqpdays`) is an actionable churn driver: customers with aging handsets are more likely to leave, and subsidized upgrades can reset lifecycle tenure while securing contract renewals.

The analytical goal is binary churn classification ranked by **precision–recall performance**, with outputs mapped to **expected revenue preserved** under conservative, base, and aggressive intervention scenarios.

## Environment Setup

Run locally from the project root (folder containing `telecom/`), or in Google Colab after mounting Drive and installing dependencies.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Google Colab: {IN_COLAB}")
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # 1. Change to your project folder
    %cd /content/drive/MyDrive/Final Assignment

    # 2. Install correct OpenCL packages + NVIDIA runtime
    %apt-get update -qq
    %apt-get install -y -qq ocl-icd-opencl-dev opencl-headers clinfo
    %apt-get install -y -qq nvidia-opencl-icd-384   # needed for NVIDIA GPU

    # 3. Rebuild LightGBM with GPU support
    %pip uninstall -y lightgbm
    %pip install lightgbm --config-settings=cmake.define.USE_GPU=ON

    # 4. Install remaining packages (already GPU-capable)
    %pip install optuna catboost xgboost scikit-learn pandas numpy scipy matplotlib seaborn pyarrow joblib

    # 5. (Optional) Verify LightGBM GPU works
    import lightgbm as lgb
    import numpy as np
    X = np.random.rand(100, 5)
    y = np.random.randint(0, 2, 100)
    try:
        model = lgb.LGBMClassifier(device_type='gpu', n_estimators=2, verbose=-1)
        model.fit(X, y)
        print("LightGBM GPU is active!")
    except Exception as e:
        print("LightGBM GPU not available:", e)
    pass
else:
    print("Local run: ensure required packages are installed.")

In [ ]:
!clinfo  # shows OpenCL platform details

In [ ]:
from __future__ import annotations

import itertools
import json
import os
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from optuna.samplers import TPESampler
from scipy import stats
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Imports loaded.")

In [ ]:
SEED = 42
TARGET = "churn"
JOIN_KEY = "Customer_ID"
BASELINE_PR_AUC = 0.682234

ROOT = Path(os.getcwd())

class Paths:
    root = ROOT
    outputs = ROOT / "outputs"
    figures = outputs / "figures"
    models = outputs / "models"
    tables = outputs / "tables"
    optuna = outputs / "optuna"
    checkpoints = outputs / "checkpoints"


def ensure_dirs() -> None:
    for p in [Paths.outputs, Paths.figures, Paths.models, Paths.tables, Paths.optuna, Paths.checkpoints]:
        p.mkdir(parents=True, exist_ok=True)


@dataclass
class TelecomConfig:
    project_root: Path = field(default_factory=lambda: ROOT)
    target: str = TARGET
    join_key: str = JOIN_KEY
    seed: int = SEED
    scenario_params: Dict[str, Dict[str, float]] = field(
        default_factory=lambda: {
            "Conservative": {"acceptance_rate": 0.20, "retained_months": 12, "subsidy_cost": 35, "marketing_cost": 10, "gross_margin_multiplier": 0.80},
            "Base": {"acceptance_rate": 0.30, "retained_months": 18, "subsidy_cost": 50, "marketing_cost": 15, "gross_margin_multiplier": 1.00},
            "Aggressive": {"acceptance_rate": 0.40, "retained_months": 24, "subsidy_cost": 65, "marketing_cost": 20, "gross_margin_multiplier": 1.10},
        }
    )

    @property
    def tables_dir(self) -> Path:
        return Paths.tables

    @property
    def figures_dir(self) -> Path:
        return Paths.figures

    @property
    def models_dir(self) -> Path:
        return Paths.models


CONFIG = TelecomConfig()
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
ensure_dirs()

client_path = ROOT / "telecom" / "Client.csv"
record_path = ROOT / "telecom" / "Record.csv"
print(f"Project root : {ROOT}")
print(f"Client.csv   : {client_path.exists()}")
print(f"Record.csv   : {record_path.exists()}")

---

## Pipeline Execution

The cells below load data, run analysis, train models, and produce business outputs. Restart the kernel and run all cells from the project root (directory containing `telecom/`).

In [ ]:
def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def safe_divide(a: pd.Series, b: pd.Series, add: float = 1.0) -> pd.Series:
    return a.fillna(0) / (b.fillna(0) + add)


def savefig_pair(stem: str, title: str) -> None:
    png = Paths.figures / f"{stem}.png"
    svg = Paths.figures / f"{stem}.svg"
    plt.tight_layout()
    plt.savefig(png, dpi=300)
    plt.savefig(svg)
    plt.close()


def select_threshold(y_true, proba) -> Tuple[float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    if len(thresholds) == 0:
        return 0.5, float(f1_score(y_true, proba >= 0.5))
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thresholds[idx]), float(f1[idx])


def metrics_for(y_true, proba, threshold) -> Dict[str, float]:
    pred = (proba >= threshold).astype(int)
    return {
        "pr_auc": float(average_precision_score(y_true, proba)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "f1": float(f1_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "threshold": float(threshold),
    }


def lift_at_decile(y_true, proba, decile: float = 0.10) -> float:
    tmp = pd.DataFrame({"y": np.asarray(y_true), "p": proba}).sort_values("p", ascending=False)
    n = max(1, int(len(tmp) * decile))
    return float(tmp.head(n)["y"].mean() / max(tmp["y"].mean(), 1e-12))


def pct_rank_against_reference(values: pd.Series, reference: pd.Series) -> pd.Series:
    ref = reference.dropna().sort_values().values
    if len(ref) == 0:
        return pd.Series(0.0, index=values.index)
    return values.apply(lambda x: float((ref <= x).mean()) if pd.notna(x) else 0.0)

In [ ]:
def load_and_merge() -> Tuple[pd.DataFrame, pd.DataFrame]:
    client = pd.read_csv(ROOT / "telecom" / "Client.csv")
    record = pd.read_csv(ROOT / "telecom" / "Record.csv")
    client.columns = client.columns.str.strip()
    record.columns = record.columns.str.strip()
    for frame in (client, record):
        if frame[JOIN_KEY].duplicated().any():
            frame["_complete"] = frame.notna().sum(axis=1)
            frame.sort_values(["_complete", JOIN_KEY], ascending=[False, True], inplace=True)
            frame.drop_duplicates(JOIN_KEY, inplace=True)
            frame.drop(columns="_complete", inplace=True)
    merged = record.merge(client, how="left", on=JOIN_KEY, validate="one_to_one")
    audit = pd.DataFrame([{
        "merged_rows": len(merged),
        "churn_rate": float(merged[TARGET].mean()),
        "target_nulls": int(merged[TARGET].isna().sum()),
    }])
    audit.to_csv(Paths.tables / "merge_audit.csv", index=False)
    return merged, audit


def quality_audit(merged: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
    for col in merged.columns:
        s = merged[col]
        miss = float(s.isna().mean())
        issue = "critical" if col == TARGET and s.isna().any() else "informational"
        if miss > 0.30:
            issue = "high"
        rows.append({
            "column": col, "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()), "missing_pct": miss,
            "distinct_count": int(s.nunique(dropna=True)), "issue_level": issue,
        })
    report = pd.DataFrame(rows)
    report.to_csv(Paths.tables / "data_quality_report.csv", index=False)

    plt.figure(figsize=(14, 6))
    missing = merged.isna().mean().sort_values(ascending=False).head(40)
    sns.barplot(x=missing.values, y=missing.index, color="#4C78A8")
    plt.title("Missingness Profile — Top Fields")
    plt.xlabel("Missing share")
    savefig_pair("01_missingness_profile", "Missingness profile")
    return report, missing.to_frame("missing_pct")


def run_eda(merged: pd.DataFrame) -> pd.DataFrame:
    key_numeric = [c for c in ["rev_Mean", "mou_Mean", "eqpdays", "custcare_Mean", "ovrmou_Mean", "phones", "months", "totmrc_Mean"] if c in merged.columns]

    plt.figure(figsize=(10, 5.6))
    ax = sns.countplot(data=merged, x=TARGET, palette=["#4C78A8", "#F58518"])
    ax.set_title("Target Class Balance")
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
    savefig_pair("02_target_balance", "Target balance")

    if "eqpdays" in merged.columns:
        bands = pd.cut(merged["eqpdays"], bins=[-np.inf, 180, 365, 730, np.inf], labels=["<180", "180-365", "365-730", "730+"])
        tmp = merged.assign(device_age_band=bands).groupby("device_age_band", observed=False)[TARGET].mean().reset_index()
        plt.figure(figsize=(10, 5.6))
        sns.barplot(data=tmp, x="device_age_band", y=TARGET, color="#59A14F")
        plt.title("Churn Rate by Device Age Band")
        savefig_pair("03_eda_device_age_churn", "Device age churn")

    for col, stem, title in [
        ("new_cell", "04_eda_new_cell_churn", "Churn by Handset Category"),
        ("custcare_Mean", "05_eda_custcare_churn", "Churn by Customer Care Intensity"),
        ("phones", "06_eda_phones_churn", "Churn by Lines per Account"),
        ("rev_Mean", "07_eda_revenue_churn", "Churn by Revenue Band"),
    ]:
        if col not in merged.columns:
            continue
        if merged[col].nunique(dropna=True) > 8:
            q = pd.qcut(merged[col].rank(method="first"), q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
            tmp = merged.assign(segment=q).groupby("segment", observed=False)[TARGET].mean().reset_index()
            xcol = "segment"
        else:
            tmp = merged.groupby(col, dropna=False)[TARGET].mean().reset_index()
            xcol = col
        plt.figure(figsize=(10, 5.6))
        sns.barplot(data=tmp, x=xcol, y=TARGET, color="#B279A2")
        plt.title(title)
        savefig_pair(stem, title)

    corr_cols = [c for c in key_numeric if merged[c].dtype.kind in "biufc"]
    if len(corr_cols) >= 2:
        plt.figure(figsize=(11, 8))
        sns.heatmap(merged[corr_cols + [TARGET]].corr(), cmap="vlag", center=0)
        plt.title("Correlation Heatmap — Key Numeric Features")
        savefig_pair("08_correlation_heatmap", "Correlation heatmap")

    stats_rows = []
    for col in key_numeric:
        a = merged.loc[merged[TARGET] == 0, col].dropna()
        b = merged.loc[merged[TARGET] == 1, col].dropna()
        if len(a) and len(b):
            ks = stats.ks_2samp(a, b)
            stats_rows.append({"feature": col, "test": "ks_2samp", "statistic": float(ks.statistic), "p_value": float(ks.pvalue)})
    stats_df = pd.DataFrame(stats_rows).sort_values("p_value")
    stats_df.to_csv(Paths.tables / "eda_statistical_checks.csv", index=False)
    return stats_df

## Data Understanding

Load and merge customer records, audit data quality, and summarize the modeling frame.

In [ ]:
print("Data discovery:")
for label, path in [("Client", client_path), ("Record", record_path)]:
    print(f"  {label}: {path.name} — exists={path.exists()}, size={path.stat().st_size if path.exists() else 0:,} bytes")

df_raw, merge_audit = load_and_merge()
print(merge_audit.to_string(index=False))
print(f"\nShape: {df_raw.shape}")
print(df_raw[TARGET].value_counts(normalize=True).rename("share").round(4))
quality_report, _ = quality_audit(df_raw)
quality_report.sort_values("missing_pct", ascending=False).head(10)

## Exploratory Data Analysis

Visualize target balance, device-age churn patterns, key behavioural drivers, correlations, and Kolmogorov–Smirnov separation tests.

In [ ]:
eda_stats = run_eda(df_raw)
print("K-S tests (lower p-value → stronger churn separation):")
eda_stats

## Feature Engineering

Row-level lifecycle, behavioural, revenue, and interaction features are created first. Train-only medians, percentile ranks, and clustering fits prevent leakage into validation and holdout sets.

In [ ]:
def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "eqpdays" in out.columns:
        out["device_age_band"] = pd.cut(
            out["eqpdays"], bins=[-np.inf, 180, 365, 730, np.inf],
            labels=["<180", "180-365", "365-730", "730+"],
        ).astype("object")
    if "phones" in out.columns:
        out["multi_line_flag"] = (out["phones"].fillna(0) > 1).astype(int)
    for col in ["eqpdays", "months", "custcare_Mean", "ovrmou_Mean", "roam_Mean", "rev_Mean", "phones", "mou_Mean", "totmrc_Mean", "ovrrev_Mean"]:
        if col not in out.columns:
            out[col] = 0
    out["eqpdays_months_interaction"] = out["eqpdays"].fillna(0) * out["months"].fillna(0)
    out["eqpdays_per_month"] = safe_divide(out["eqpdays"], out["months"])
    out["custcare_per_month"] = safe_divide(out["custcare_Mean"], out["months"])
    out["overage_per_month"] = safe_divide(out["ovrmou_Mean"], out["months"])
    out["roaming_per_month"] = safe_divide(out["roam_Mean"], out["months"])
    out["revenue_per_phone"] = safe_divide(out["rev_Mean"], out["phones"])
    out["revenue_efficiency"] = safe_divide(out["rev_Mean"], out["mou_Mean"])
    out["rev_per_mou"] = safe_divide(out["rev_Mean"], out["mou_Mean"])
    out["overage_intensity"] = safe_divide(out["ovrrev_Mean"], out["totmrc_Mean"])
    out["roam_ratio"] = safe_divide(out["roam_Mean"], out["mou_Mean"])
    out["custcare_intensity"] = out["custcare_Mean"].fillna(0) * out["months"].fillna(0)
    out["clv_proxy"] = out["rev_Mean"].fillna(0) * out["months"].fillna(0)
    out["device_age_customer_care"] = out["eqpdays"].fillna(0) * out["custcare_Mean"].fillna(0)
    out["overage_tenure"] = out["ovrmou_Mean"].fillna(0) * out["months"].fillna(0)
    out["revenue_customer_care"] = out["rev_Mean"].fillna(0) * out["custcare_Mean"].fillna(0)
    if "custcare_Mean" in out.columns and "overage_intensity" in out.columns:
        out["satisfaction_proxy"] = 1 / (1 + out["custcare_Mean"].fillna(0) + out["overage_intensity"].fillna(0))
    return out


def apply_train_derived_features(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    median_eqpdays = float(train["eqpdays"].median())
    for part in (train, val, test):
        part["device_age_relative"] = part["eqpdays"].fillna(median_eqpdays) / max(median_eqpdays, 1)
        part["customer_value_score"] = (
            pct_rank_against_reference(part["rev_Mean"], train["rev_Mean"])
            + pct_rank_against_reference(part["months"], train["months"])
            + pct_rank_against_reference(part["phones"], train["phones"])
        ) / 3

    cluster_cols = [
        "eqpdays", "months", "custcare_Mean", "ovrmou_Mean", "roam_Mean",
        "rev_Mean", "mou_Mean", "totmrc_Mean", "phones",
        "customer_value_score", "revenue_efficiency", "overage_per_month",
    ]
    cluster_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    train_matrix = cluster_pipe.fit_transform(train[cluster_cols].replace([np.inf, -np.inf], np.nan))
    kmeans = KMeans(n_clusters=5, random_state=SEED, n_init=20)
    gmm = GaussianMixture(n_components=5, random_state=SEED, covariance_type="diag", max_iter=150)
    kmeans.fit(train_matrix)
    gmm.fit(train_matrix)
    for part in (train, val, test):
        matrix = cluster_pipe.transform(part[cluster_cols].replace([np.inf, -np.inf], np.nan))
        part["kmeans_segment"] = kmeans.predict(matrix).astype(str)
        part["gmm_segment"] = gmm.predict(matrix).astype(str)

    joblib.dump({"pipeline": cluster_pipe, "kmeans": kmeans, "gmm": gmm, "columns": cluster_cols}, Paths.models / "customer_segmentation_models.joblib")

    segment_profile = (
        train.groupby(["kmeans_segment", "gmm_segment"], observed=False)
        .agg(customers=(TARGET, "size"), churn_rate=(TARGET, "mean"), avg_revenue=("rev_Mean", "mean"), avg_device_age=("eqpdays", "mean"))
        .reset_index().sort_values(["churn_rate", "avg_revenue"], ascending=False)
    )
    return train, val, test, segment_profile


FEATURE_DICT = pd.DataFrame([
    ("device_age_band", "bin(eqpdays)", "Lifecycle replacement window"),
    ("multi_line_flag", "phones > 1", "Household stickiness"),
    ("eqpdays_per_month", "eqpdays / (months+1)", "Device age vs tenure"),
    ("custcare_per_month", "custcare / (months+1)", "Service friction rate"),
    ("customer_value_score", "train-percentile composite", "Economic value ranking"),
    ("kmeans_segment", "KMeans k=5 on train", "Behaviour segment"),
    ("gmm_segment", "GMM k=5 on train", "Probabilistic segment"),
], columns=["feature", "formula", "business_meaning"])

### Customer Segment Profiles

KMeans and GMM segments (fit on training data) summarize churn and revenue patterns by cohort.

In [ ]:
df_feat = add_base_features(df_raw)
y = df_feat[TARGET].astype(int)
X = df_feat.drop(columns=[TARGET])
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=SEED)

train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
val_df = pd.concat([X_val.reset_index(drop=True), y_val.reset_index(drop=True)], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

train_df, val_df, test_df, segment_profile = apply_train_derived_features(train_df, val_df, test_df)
df_model = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)

FEATURE_DICT.to_csv(Paths.tables / "advanced_feature_dictionary.csv", index=False)
segment_profile.to_csv(Paths.tables / "customer_segment_profiles.csv", index=False)
print(f"Modeling frame: {df_model.shape[1]} columns, {len(df_model):,} rows")
segment_profile.head(8)

## Preprocessing

A stratified 64/16/20 train/validation/test split is applied. The column transformer (median imputation with missing indicators, one-hot encoding) is **fit on training data only** before transforming validation and holdout sets.

In [ ]:
def make_preprocessor(df: pd.DataFrame):
    X = df.drop(columns=[TARGET, JOIN_KEY], errors="ignore")
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=25, sparse_output=True)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore")
    preprocessor = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("scaler", StandardScaler(with_mean=False))]), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)]), categorical_cols),
    ], sparse_threshold=0.35)
    return preprocessor, numeric_cols, categorical_cols


def matrices_from_splits(train_df, val_df, test_df, train_full_df):
    preprocessor, num_cols, cat_cols = make_preprocessor(train_full_df)
    drop = [JOIN_KEY]
    Xt_train = preprocessor.fit_transform(train_df.drop(columns=[TARGET] + drop, errors="ignore"))
    Xt_val = preprocessor.transform(val_df.drop(columns=[TARGET] + drop, errors="ignore"))
    Xt_test = preprocessor.transform(test_df.drop(columns=[TARGET] + drop, errors="ignore"))
    Xt_train_full = preprocessor.transform(train_full_df.drop(columns=[TARGET] + drop, errors="ignore"))
    feature_names = preprocessor.get_feature_names_out().tolist()
    joblib.dump(preprocessor, Paths.models / "advanced_preprocessor.joblib")
    save_json(Paths.tables / "advanced_split_metadata.json", {
        "train_rows": len(train_df), "validation_rows": len(val_df), "test_rows": len(test_df),
        "feature_count": len(feature_names),
    })
    y_train = train_df[TARGET].astype(int)
    y_val = val_df[TARGET].astype(int)
    y_test = test_df[TARGET].astype(int)
    y_train_full = train_full_df[TARGET].astype(int)
    return (
        train_full_df.drop(columns=[TARGET]), train_df.drop(columns=[TARGET]), val_df.drop(columns=[TARGET]), test_df.drop(columns=[TARGET]),
        y_train_full, y_train, y_val, y_test,
        Xt_train_full, Xt_train, Xt_val, Xt_test, feature_names,
    )

train_full_df = pd.concat([train_df, val_df], axis=0, ignore_index=True)
(
    X_train_full, X_train, X_val, X_test,
    y_train_full, y_train, y_val, y_test,
    Xt_train_full, Xt_train, Xt_val, Xt_test, feature_names,
) = matrices_from_splits(train_df, val_df, test_df, train_full_df)

print(f"Train / Val / Test features: {Xt_train.shape}, {Xt_val.shape}, {Xt_test.shape}")
print(f"Churn rates — train {y_train.mean():.4f}, val {y_val.mean():.4f}, test {y_test.mean():.4f}")

## Modeling

Five classifiers are benchmarked with 5-fold stratified cross-validation. LightGBM is tuned with 50 Optuna trials, then blended into a validation-weighted ensemble.

In [ ]:
def candidate_models(scale_pos_weight: float) -> Dict[str, Any]:
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=SEED),
        "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="aucpr", random_state=SEED,
            tree_method="hist", device="cuda",
            n_estimators=450, max_depth=4, learning_rate=0.045, subsample=0.88, colsample_bytree=0.90,
            min_child_weight=3, gamma=0.25, reg_alpha=0.1, reg_lambda=1.8, scale_pos_weight=scale_pos_weight),
        "LightGBM": LGBMClassifier(objective="binary", random_state=SEED, n_estimators=550, learning_rate=0.045,
            device_type="gpu",
            num_leaves=48, subsample=0.88, colsample_bytree=0.88, reg_alpha=0.1, reg_lambda=1.2, class_weight="balanced", verbosity=-1),
        "CatBoost": CatBoostClassifier(iterations=450, learning_rate=0.045, depth=6, loss_function="Logloss",
            task_type="GPU",
            random_seed=SEED, auto_class_weights="Balanced", verbose=False, allow_writing_files=False),
        "RandomForest": RandomForestClassifier(n_estimators=220, max_depth=16, min_samples_leaf=20,
            class_weight="balanced_subsample", random_state=SEED, n_jobs=-1),
    }

def benchmark_models(Xt_train_full, y_train_full, Xt_test, y_test, Xt_val, y_val):
    scale_pos_weight = float((y_train_full == 0).sum() / max((y_train_full == 1).sum(), 1))
    models = candidate_models(scale_pos_weight)
    folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(Xt_train_full, y_train_full))
    fold_rows, fitted = [], {}
    for name, model in models.items():
        for fold_id, (tr, va) in enumerate(folds, start=1):
            m = clone(model)
            m.fit(Xt_train_full[tr], np.asarray(y_train_full)[tr])
            proba = m.predict_proba(Xt_train_full[va])[:, 1]
            thr, _ = select_threshold(np.asarray(y_train_full)[va], proba)
            row = metrics_for(np.asarray(y_train_full)[va], proba, thr)
            row.update({"model": name, "fold": fold_id})
            fold_rows.append(row)
        final = clone(model)
        final.fit(Xt_train_full, y_train_full)
        fitted[name] = final
        joblib.dump(final, Paths.models / f"advanced_{name}.joblib")
    fold_df = pd.DataFrame(fold_rows)
    summary = fold_df.groupby("model").agg(
        cv_pr_auc_mean=("pr_auc", "mean"), cv_pr_auc_std=("pr_auc", "std"),
        cv_roc_auc_mean=("roc_auc", "mean"), cv_f1_mean=("f1", "mean"),
    ).reset_index()
    test_rows = []
    for name, model in fitted.items():
        thr, _ = select_threshold(y_val, model.predict_proba(Xt_val)[:, 1])
        row = metrics_for(y_test, model.predict_proba(Xt_test)[:, 1], thr)
        row["model"] = name
        test_rows.append(row)
    test_df = pd.DataFrame(test_rows).rename(columns={
        "pr_auc": "test_pr_auc", "roc_auc": "test_roc_auc", "f1": "test_f1",
        "precision": "test_precision", "recall": "test_recall", "threshold": "test_threshold",
    })
    out = summary.merge(test_df, on="model").sort_values("test_pr_auc", ascending=False)
    fold_df.to_csv(Paths.tables / "model_benchmark_expanded_folds.csv", index=False)
    out.to_csv(Paths.tables / "model_benchmark_expanded.csv", index=False)
    return out, fitted


benchmark, fitted_models = benchmark_models(Xt_train_full, y_train_full, Xt_test, y_test, Xt_val, y_val)
benchmark[["model", "cv_pr_auc_mean", "test_pr_auc", "test_roc_auc", "test_f1", "test_precision", "test_recall"]]

In [ ]:
def run_optuna(Xt_train, y_train, Xt_val, y_val, Xt_train_full, y_train_full, Xt_test, y_test):
    study_db = Paths.optuna / "optuna_lgbm_study.db"
    study = optuna.create_study(
        study_name="lgbm_pr_auc", direction="maximize", sampler=TPESampler(seed=SEED),
        storage=f"sqlite:///{study_db.as_posix()}", load_if_exists=True,
    )

    def objective(trial):
        params = {
            "objective": "binary", "random_state": SEED, "verbosity": -1, "class_weight": "balanced",
            "device_type": "gpu", # <--- GPU FLAG ADDED
            "n_estimators": trial.suggest_int("n_estimators", 350, 1200),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 24, 128),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 220),
            "subsample": trial.suggest_float("subsample", 0.65, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }
        model = LGBMClassifier(**params)
        model.fit(Xt_train, y_train, eval_set=[(Xt_val, y_val)], eval_metric="average_precision")
        return float(average_precision_score(y_val, model.predict_proba(Xt_val)[:, 1]))

    remaining = max(0, 50 - len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]))
    if remaining:
        study.optimize(objective, n_trials=remaining, show_progress_bar=False)

    best_params = dict(study.best_params)
    # ADDED GPU FLAG BELOW
    best_params.update({"objective": "binary", "random_state": SEED, "class_weight": "balanced", "verbosity": -1, "device_type": "gpu"})

    final_model = LGBMClassifier(**best_params)
    final_model.fit(Xt_train_full, y_train_full)
    val_model = LGBMClassifier(**best_params)
    val_model.fit(Xt_train, y_train)
    val_proba = val_model.predict_proba(Xt_val)[:, 1]
    threshold, val_f1 = select_threshold(y_val, val_proba)
    test_proba = final_model.predict_proba(Xt_test)[:, 1]
    metrics = metrics_for(y_test, test_proba, threshold)
    metrics.update({
        "model": "OptunaLightGBM",
        "validation_pr_auc": float(average_precision_score(y_val, val_proba)),
        "validation_f1": val_f1,
        "completed_trials": len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
    })
    joblib.dump(final_model, Paths.models / "optuna_lightgbm.joblib")
    trials_df = study.trials_dataframe()
    if not trials_df.empty:
        completed = trials_df[trials_df["state"] == "COMPLETE"]
        plt.figure(figsize=(10, 5.6))
        plt.plot(range(1, len(completed) + 1), completed["value"].cummax(), color="#4C78A8")
        plt.xlabel("Completed trial"); plt.ylabel("Best validation PR-AUC")
        plt.title("Optuna LightGBM Optimization History")
        savefig_pair("18_optuna_optimization_history", "Optuna history")
    pd.DataFrame([metrics]).to_csv(Paths.tables / "optuna_final_metrics.csv", index=False)
    trials_df.to_csv(Paths.optuna / "optuna_trials.csv", index=False)
    return metrics, final_model, test_proba, best_params


optuna_metrics, optuna_model, _, best_params = run_optuna(
    Xt_train, y_train, Xt_val, y_val, Xt_train_full, y_train_full, Xt_test, y_test
)
pd.Series(optuna_metrics)

In [ ]:
def build_validation_weighted_ensemble(Xt_train, y_train, Xt_val, y_val, Xt_test, y_test, Xt_train_full, y_train_full):
    scale_pos_weight = float((y_train_full == 0).sum() / max((y_train_full == 1).sum(), 1))
    base_models = candidate_models(scale_pos_weight)
    selected = ["XGBoost", "LightGBM", "CatBoost"]
    val_probas, test_probas = {}, {}
    for name in selected:
        m = clone(base_models[name])
        m.fit(Xt_train, y_train)
        val_probas[name] = m.predict_proba(Xt_val)[:, 1]
        full = joblib.load(Paths.models / f"advanced_{name}.joblib")
        test_probas[name] = full.predict_proba(Xt_test)[:, 1]

    optuna_val = LGBMClassifier(**best_params)
    optuna_val.fit(Xt_train, y_train)
    val_probas["OptunaLightGBM"] = optuna_val.predict_proba(Xt_val)[:, 1]
    test_probas["OptunaLightGBM"] = optuna_model.predict_proba(Xt_test)[:, 1]

    names = list(val_probas)
    best_score, best_weights = -np.inf, None
    for weights_int in itertools.product(range(21), repeat=len(names)):
        if sum(weights_int) != 20 or sum(w > 0 for w in weights_int) < 2:
            continue
        weights = np.array(weights_int, dtype=float) / 20.0
        blend = sum(weights[i] * val_probas[names[i]] for i in range(len(names)))
        score = average_precision_score(y_val, blend)
        if score > best_score:
            best_score, best_weights = score, weights

    val_blend = sum(best_weights[i] * val_probas[names[i]] for i in range(len(names)))
    test_blend = sum(best_weights[i] * test_probas[names[i]] for i in range(len(names)))
    threshold, val_f1 = select_threshold(y_val, val_blend)
    metrics = metrics_for(y_test, test_blend, threshold)
    weights_dict = {names[i]: float(best_weights[i]) for i in range(len(names)) if best_weights[i] > 0}
    metrics.update({
        "model": "ValidationWeightedEnsemble",
        "validation_pr_auc": float(best_score),
        "validation_f1": float(val_f1),
        "weights": json.dumps(weights_dict),
        "baseline_pr_auc": BASELINE_PR_AUC,
        "improvement_pct_vs_baseline": float((metrics["pr_auc"] - BASELINE_PR_AUC) / BASELINE_PR_AUC * 100),
    })
    pd.DataFrame([metrics]).to_csv(Paths.tables / "ensemble_metrics.csv", index=False)
    pd.DataFrame({"y_test": np.asarray(y_test), "p_churn": test_blend}).to_parquet(Paths.checkpoints / "ensemble_predictions.parquet", index=False)
    joblib.dump({"weights": weights_dict, "threshold": threshold}, Paths.models / "weighted_ensemble.joblib")
    return metrics, test_blend, threshold, weights_dict


ensemble_metrics, champion_proba, champion_threshold, ensemble_weights = build_validation_weighted_ensemble(
    Xt_train, y_train, Xt_val, y_val, Xt_test, y_test, Xt_train_full, y_train_full
)
print("Ensemble weights:", ensemble_weights)
for k, v in ensemble_metrics.items():
    print(f"  {k}: {v}")

## Evaluation

Holdout metrics, precision–recall and ROC curves, confusion matrix, and decile lift for the champion ensemble.

In [ ]:
eval_df = pd.DataFrame([{
    "model": ensemble_metrics["model"],
    "pr_auc": ensemble_metrics["pr_auc"],
    "roc_auc": ensemble_metrics["roc_auc"],
    "f1": ensemble_metrics["f1"],
    "precision": ensemble_metrics["precision"],
    "recall": ensemble_metrics["recall"],
    "threshold": ensemble_metrics["threshold"],
    "lift_at_top_decile": lift_at_decile(y_test, champion_proba),
}])
eval_df.to_csv(Paths.tables / "champion_holdout_metrics.csv", index=False)
print(eval_df.to_string(index=False))

precision_vals, recall_vals, _ = precision_recall_curve(y_test, champion_proba)
fpr, tpr, _ = roc_curve(y_test, champion_proba)
plt.figure(figsize=(10, 5.6))
plt.plot(recall_vals, precision_vals, color="#4C78A8")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–Recall Curve — Champion Ensemble")
savefig_pair("09_model_precision_recall", "PR curve")
plt.figure(figsize=(10, 5.6))
plt.plot(fpr, tpr, color="#4C78A8")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title("ROC Curve — Champion Ensemble")
savefig_pair("10_model_roc_curve", "ROC curve")

pred = (champion_proba >= champion_threshold).astype(int)
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Holdout Confusion Matrix")
savefig_pair("11_confusion_matrix", "Confusion matrix")

lift_rows = []
base_rate = float(np.mean(y_test))
tmp = pd.DataFrame({"y": y_test, "p": champion_proba}).sort_values("p", ascending=False)
for pct in [0.05, 0.10, 0.20, 0.30]:
    n = max(1, int(len(tmp) * pct))
    rate = float(tmp.head(n)["y"].mean())
    lift_rows.append({"decile_pct": pct, "capture_rate": rate, "lift": rate / max(base_rate, 1e-12)})
lift_df = pd.DataFrame(lift_rows)
plt.figure(figsize=(8, 5))
sns.barplot(data=lift_df, x="decile_pct", y="lift", color="#59A14F")
plt.title("Lift vs Random Targeting")
plt.xlabel("Top fraction targeted"); plt.ylabel("Lift")
savefig_pair("12_lift_chart", "Lift chart")
lift_df

## Model Interpretation

Global feature importance from the tuned LightGBM component identifies the operational drivers of churn risk.

In [ ]:
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": optuna_model.feature_importances_,
}).sort_values("importance", ascending=False)
importance.head(25).to_csv(Paths.tables / "feature_importance.csv", index=False)
plt.figure(figsize=(11, 7))
sns.barplot(data=importance.head(20), y="feature", x="importance", color="#59A14F")
plt.title("Top Feature Importances — Optuna LightGBM")
savefig_pair("19_feature_importance", "Feature importance")
importance.head(15)

## Business Impact

Churn probabilities are converted into three ROI scenarios targeting the top decile by expected net value. Assumptions: handset subsidy + marketing cost per customer, acceptance rate, and retained contract months.

In [ ]:
def business_impact(df_test_frame, y_test, proba, threshold):
    scored = df_test_frame.copy().reset_index(drop=True)
    scored["actual_churn"] = np.asarray(y_test)
    scored["p_churn"] = proba
    scored["clv_proxy"] = scored.get("clv_proxy", scored["rev_Mean"].fillna(0) * scored["months"].fillna(0))
    scenarios = []
    for name, params in CONFIG.scenario_params.items():
        cost = params["subsidy_cost"] + params["marketing_cost"]
        scored[f"{name}_expected_preserved_value"] = (
            scored["p_churn"] * params["acceptance_rate"] * params["retained_months"]
            * scored["rev_Mean"].fillna(0) * params["gross_margin_multiplier"]
        )
        scored[f"{name}_expected_campaign_cost"] = cost
        scored[f"{name}_expected_net_value"] = scored[f"{name}_expected_preserved_value"] - cost
        targets = scored.sort_values([f"{name}_expected_net_value", "p_churn", "rev_Mean"], ascending=False).head(max(1, int(len(scored) * 0.10)))
        scenarios.append({
            "scenario": name,
            "targeted_customers": len(targets),
            "gross_preserved_value": float(targets[f"{name}_expected_preserved_value"].sum()),
            "campaign_cost": float(targets[f"{name}_expected_campaign_cost"].sum()),
            "net_value": float(targets[f"{name}_expected_net_value"].sum()),
            "roi": float(targets[f"{name}_expected_net_value"].sum() / max(targets[f"{name}_expected_campaign_cost"].sum(), 1e-12)),
            "lift_over_random": lift_at_decile(y_test, proba),
        })
    summary = pd.DataFrame(scenarios)
    ranked = scored.sort_values(["Base_expected_net_value", "p_churn", "rev_Mean"], ascending=False)
    summary.to_csv(Paths.tables / "business_impact_summary.csv", index=False)
    ranked.head(1000).to_csv(Paths.tables / "top_risk_customers.csv", index=False)
    plt.figure(figsize=(10, 5.6))
    sns.barplot(data=summary, x="scenario", y="net_value", palette=["#4C78A8", "#59A14F", "#F58518"])
    plt.title("Expected Net Value by Retention Scenario")
    savefig_pair("16_business_roi_waterfall", "ROI scenarios")
    plt.figure(figsize=(10, 5.6))
    sns.scatterplot(
        data=scored.head(5000), x="p_churn", y="clv_proxy",
        hue="Base_expected_net_value", palette="viridis", s=18, legend=False,
    )
    plt.axvline(threshold, color="red", linestyle="--")
    plt.title("Retention Priority: Churn Risk vs Customer Value")
    plt.xlabel("Predicted churn probability"); plt.ylabel("CLV proxy")
    savefig_pair("17_priority_scatter", "Priority scatter")
    return ranked, summary


scored_customers, impact_summary = business_impact(test_df.drop(columns=[TARGET]), y_test, champion_proba, champion_threshold)
print("Top 10 priority customers (holdout):")
scored_customers[["p_churn", "rev_Mean", "eqpdays", "custcare_Mean", "Base_expected_net_value"]].head(10)
impact_summary

## Final Summary

### Recommendation

Deploy the **ValidationWeightedEnsemble** to score customers monthly. Prioritize the top decile by expected net value for device-upgrade offers when `eqpdays` exceeds 365 days, layered with service recovery for high `custcare_Mean` accounts.

### Grading Summary

| Criterion | Evidence in this notebook |
|-----------|---------------------------|
| Data integration | `Client.csv` + `Record.csv` merged on `Customer_ID`; ID dropped before modeling |
| Data quality | Missingness audit (Fig 01), column-level quality report |
| EDA | Target balance, device-age churn, correlation heatmap, K-S tests (Figs 02–08) |
| Feature engineering | Lifecycle, behavioural, revenue, interaction, train-fitted segmentation |
| Modeling | LR, RF, XGBoost, LightGBM, CatBoost benchmark; Optuna tuning; weighted ensemble |
| Evaluation | PR-AUC primary; ROC-AUC, F1, precision, recall, lift, confusion matrix |
| Business translation | Top-risk customers, three ROI scenarios, actionable upgrade recommendation |

### Champion Results

| Metric | Value |
|--------|-------|
| Model | ValidationWeightedEnsemble |
| Holdout PR-AUC | 0.685036 |
| Holdout ROC-AUC | 0.695620 |
| F1 | 0.689465 |
| Precision | 0.554893 |
| Recall | 0.910210 |
| Weights | OptunaLightGBM 80% · LightGBM 15% · CatBoost 5% |

### Conclusion

Device lifecycle intelligence combined with gradient-boosted ensemble scoring delivers measurable lift over baseline retention targeting. The expected net value framework ties model outputs directly to subsidized-upgrade campaign economics, giving Company A a defensible, data-driven retention playbook.

In [ ]:
print("=" * 72)
print("SUBMISSION GRADING SUMMARY")
print("=" * 72)
print(f"Champion model      : {ensemble_metrics['model']}")
print(f"Holdout PR-AUC      : {ensemble_metrics['pr_auc']:.6f}")
print(f"Holdout ROC-AUC     : {ensemble_metrics['roc_auc']:.6f}")
print(f"F1                  : {ensemble_metrics['f1']:.6f}")
print(f"Precision           : {ensemble_metrics['precision']:.6f}")
print(f"Recall              : {ensemble_metrics['recall']:.6f}")
print(f"Decision threshold  : {ensemble_metrics['threshold']:.6f}")
print(f"Lift @ top 10%      : {lift_at_decile(y_test, champion_proba):.3f}x")
print(f"Ensemble weights    : {ensemble_weights}")
print("-" * 72)
print("Business impact (Base scenario):")
base = impact_summary.loc[impact_summary["scenario"] == "Base"].iloc[0]
print(f"  Targeted customers : {int(base['targeted_customers']):,}")
print(f"  Net value          : ${base['net_value']:,.0f}")
print(f"  ROI                : {base['roi']:.2f}x")
print("=" * 72)